In [1]:
import geogridfusion
import pvdeg
import pandas as pd

In [2]:
conn = geogridfusion.start()

Starting Postgres subprocess...
PostgreSQL connection established after 2.20 seconds.
postgis already installed


In [3]:
geogridfusion.initialize_tables(conn)

In [8]:
client = pvdeg.geospatial.start_dask()

coords = [(45 + i, -115 + k) for i in range(5) for k in range(5)]

geo_weather, geo_meta, failed = pvdeg.weather.weather_distributed(
    database="PVGIS",
    coords=coords
)

client.close()

Dashboard: http://127.0.0.1:8787/status
Connected to a Dask scheduler | Dashboard: http://127.0.0.1:8787/status


In [9]:
geo_meta

,latitude,longitude,irradiance_time_offset,altitude,wind_height,Source
0,45.0,-115.0,0.0,1948.0,10,PVGIS
1,45.0,-114.0,0.0,2076.0,10,PVGIS
2,45.0,-113.0,0.0,1729.0,10,PVGIS
3,45.0,-112.0,0.0,2219.0,10,PVGIS
4,45.0,-111.0,0.0,2370.0,10,PVGIS
5,46.0,-115.0,0.0,2000.0,10,PVGIS
6,46.0,-114.0,0.0,1755.0,10,PVGIS
7,46.0,-113.0,0.0,1986.0,10,PVGIS
8,46.0,-112.0,0.0,2101.0,10,PVGIS
9,46.0,-111.0,0.0,1993.0,10,PVGIS


In [4]:
weather, meta = pvdeg.weather.get(
    database="PVGIS",
    id=(45,-115)
)

In [5]:
geogridfusion.store_single(conn=conn, weather_df=weather, meta=meta, tmy=True, source_res="pvgis")

coercing tmy data to year 1979


When we run the cell below, we see that we will get a collsion from the data inserted above.

We want to rewrite this function so it can be done async or with multiprocessing.

In [ ]:
for i in range(25):

    w = geo_weather.isel(gid=i).drop_vars(("gid",)).to_pandas()
    m = geo_meta.iloc[i].to_dict()

    geogridfusion.store_single(conn=conn, weather_df=w, meta=m, tmy=True, source_res="pvgis")

coercing tmy data to year 1979
duplicate file detected, skipping insert
metadata of duplicate file {'latitude': 45.0, 'longitude': -115.0, 'irradiance_time_offset': 0.0, 'altitude': 1948.0, 'wind_height': 10, 'Source': 'PVGIS'}
coercing tmy data to year 1979
duplicate file detected, skipping insert
metadata of duplicate file {'latitude': 45.0, 'longitude': -114.0, 'irradiance_time_offset': 0.0, 'altitude': 2076.0, 'wind_height': 10, 'Source': 'PVGIS'}
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data to year 1979
coercing tmy data

We can easily get one output from what we have written so far. We need to be able to read MANY into dataset form.

In [18]:
lw, lm = geogridfusion.load_single(conn=conn, latitude=40, longitude=-110, source_name="PVGIS")

lw

,temp_air,relative_humidity,ghi,dni,dhi,IR(h),wind_speed,wind_direction,pressure
time,,,,,,,,,
1979-01-01 00:00:00,-8.82,85.54,0.0,0.0,0.0,255.83,2.42,213.0,76049.0
1979-01-01 01:00:00,-9.17,85.94,0.0,0.0,0.0,253.89,2.40,212.0,76072.0
1979-01-01 02:00:00,-9.51,86.34,0.0,0.0,0.0,251.96,2.38,215.0,76087.0
1979-01-01 03:00:00,-9.86,86.74,0.0,0.0,0.0,250.02,2.35,219.0,76109.0
1979-01-01 04:00:00,-10.21,87.14,0.0,0.0,0.0,248.08,2.33,219.0,76109.0
...,...,...,...,...,...,...,...,...,...
1979-12-31 19:00:00,-7.09,83.53,10.0,0.0,10.0,265.52,2.53,212.0,76701.0
1979-12-31 20:00:00,-7.43,83.93,13.0,0.0,13.0,263.58,2.51,215.0,76596.0
1979-12-31 21:00:00,-7.78,84.33,25.0,0.0,25.0,261.64,2.49,217.0,76559.0
